# 07 — Comparison and ablations (Stage F)

One comparison table for the three arms (classical GB @ 224, CNN, hybrid; classical GB @ 512 as
the practical-deployment configuration), bootstrap intervals on val, and three ablations, each
changing one thing from the Stage D CNN:

1. training flaw labels at >=1 vote instead of >=2 (classical GB and CNN), scored at >=2;
2. random initialisation instead of ImageNet weights (CNN);
3. frozen backbone instead of full fine-tuning (CNN, Stage D runs).

The backbone comparison (EfficientNet-B0 vs ResNet-50) is not run. Apart from the classical GB
refit at >=1 vote (seconds), this notebook trains nothing; it reads saved val scores and training
histories and regenerates `reports/results_comparison.md` and `reports/results_comparison.csv`.

**Partitions:** every number is on the val partition. The test partition is never read.

**How to run:** from the `notebooks/` folder, with `DATA_DIR` pointing at the data directory
(defaults to `../data`). `DATA_DIR` must contain `annotations/{train,val}.json`,
`classical_features_224.csv`, the Stage C to E prediction files in `predictions/`,
`predictions/cnn_finetune_{votes1,scratch}_val.csv` and the matching `cnn_runs/*/history.json`
(from `scripts/train_cnn_ablations.py`). No GPU needed; about a minute, mostly the bootstrap.

In [2]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
DATA_DIR = Path(os.environ.get("DATA_DIR", REPO_DIR / "data"))
REPORT_PATH = REPO_DIR / "reports" / "results_comparison.md"
RESULTS_CSV = REPO_DIR / "reports" / "results_comparison.csv"
PREDICTIONS_DIR = DATA_DIR / "predictions"
RUNS_DIR = DATA_DIR / "cnn_runs"

from src.data.splits import MODELLED_FLAWS, SEED, get_splits
from src.eda import report as rp
from src.models import classical as cm
from src.models import metrics as mt

N_RESAMPLES = 1000   # bootstrap resamples of the val images, seed 42
SEEDS = (42, 43, 44)
CLASSICAL, CLASSICAL_512 = "Classical GB @ 224", "Classical GB @ 512"
CNN, HYBRID, FROZEN = "CNN fine-tuned", "Hybrid", "CNN frozen"
CNN_VOTES1, CNN_SCRATCH, CLASSICAL_VOTES1 = "CNN, flaws >= 1 vote", "CNN, random init", "Classical GB @ 224, flaws >= 1 vote"
PREDICTION_FILES = {
    CLASSICAL: "classical_gb_224_val.csv", CLASSICAL_512: "classical_gb_512_val.csv",
    FROZEN: "cnn_frozen_val.csv", CNN_VOTES1: "cnn_finetune_votes1_val.csv",
    CNN_SCRATCH: "cnn_finetune_scratch_val.csv",
    **{(arm, s): f"{pred}_finetune{'' if s == 42 else f'_seed{s}'}_val.csv"
       for arm, pred in ((CNN, "cnn"), (HYBRID, "hybrid")) for s in SEEDS},
}
RUN_DIRS = {FROZEN: "frozen", CNN: "finetune", CNN_VOTES1: "finetune_votes1", CNN_SCRATCH: "finetune_scratch"}
FLAW_TYPE = {"BLR": "photometric", "DRK": "photometric", "BRT": "photometric",
             "FRM": "semantic", "OBS": "semantic", "ROT": "semantic", cm.UNREC: "recognisability"}
ROWS = cm.OUTPUTS + [mt.MACRO]
ENVIRONMENT = rp.environment_lines(
    gpu_note="not needed here: this notebook scores saved predictions and refits one gradient-boosting "
             "baseline; training GPUs are recorded in the Stage D/E reports and below",
    seed_note=f"bootstrap ({N_RESAMPLES} resamples) and gradient boosting use seed 42; CNN entries seed 42")
pd.set_option("display.precision", 3)

print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)
print("\n".join(ENVIRONMENT))

REPO_DIR: /content/drive/MyDrive/Masters/Sem 3/capstone/repo
DATA_DIR: /content/drive/MyDrive/Masters/Sem 3/capstone/data
- Python 3.13.18 on Linux-6.6.159+-x86_64-with-glibc2.39
- RAM: 12.7 GB
- GPU: none (not needed here: this notebook scores saved predictions and refits one gradient-boosting baseline; training GPUs are recorded in the Stage D/E reports and below)
- Libraries: numpy 2.4.6, pandas 2.2.3, scikit-learn 1.6.1, torch 2.14.1+cpu, torchvision 0.29.1+cpu
- Seed: 42 (bootstrap (1000 resamples) and gradient boosting use seed 42; CNN entries seed 42)


## 1. Data
Val targets at >=2 votes from `src/data/splits.py`. Every prediction file must hold exactly the
val images, one score column per output.

In [3]:
splits = get_splits(DATA_DIR)
targets = cm.build_targets(splits["val"])


def load_scores(filename):
    s = pd.read_csv(PREDICTIONS_DIR / filename, index_col=0)  # first column: image name
    s.index.name = "image"
    assert sorted(s.index) == sorted(targets.index), f"{filename}: image set differs from val"
    assert list(s.columns) == cm.OUTPUTS, f"{filename}: columns {list(s.columns)}"
    print(f"{filename:<34} {len(s):,} images")
    return s.loc[targets.index]


scores = {name: load_scores(f) for name, f in PREDICTION_FILES.items()}
scores[CNN], scores[HYBRID] = scores[CNN, 42], scores[HYBRID, 42]   # arm entries
sections = []  # (heading, markdown) pairs collected for the report


def evaluate_with_gain(s):
    table = mt.evaluate(targets, s)
    table.insert(table.columns.get_loc("AP") + 1, "gain", mt.normalised_gain(table["AP"], table["base rate"]))
    table.loc[mt.MACRO, "gain"] = np.nan  # gain is read per output, not on the averaged row
    return table


results = {name: evaluate_with_gain(s) for name, s in scores.items()}
base_rate = results[CNN].loc[ROWS, "base rate"]

classical_gb_224_val.csv           3,875 images
classical_gb_512_val.csv           3,875 images
cnn_frozen_val.csv                 3,875 images
cnn_finetune_votes1_val.csv        3,875 images
cnn_finetune_scratch_val.csv       3,875 images
cnn_finetune_val.csv               3,875 images
cnn_finetune_seed43_val.csv        3,875 images
cnn_finetune_seed44_val.csv        3,875 images
hybrid_finetune_val.csv            3,875 images
hybrid_finetune_seed43_val.csv     3,875 images
hybrid_finetune_seed44_val.csv     3,875 images


## 2. The comparison table
Flaws down the side, arms across. CNN and hybrid: the seed-42 entry plus the mean and range over
seeds 42-44 (run-to-run spread). Classical GB @ 512 is the practical-deployment configuration,
not one of the matched-input arms.

In [4]:
def seed_stats(arm):
    aps = pd.concat([results[arm, s].loc[ROWS, "AP"] for s in SEEDS], axis=1).astype(float)
    return aps.mean(axis=1), aps.min(axis=1), aps.max(axis=1)


comparison = pd.DataFrame({"type": [FLAW_TYPE.get(o, "") for o in ROWS], "base rate": base_rate}, index=ROWS)
comparison[CLASSICAL] = results[CLASSICAL].loc[ROWS, "AP"]
for arm in (CNN, HYBRID):
    mean, low, high = seed_stats(arm)
    comparison[f"{arm} (seed 42)"] = results[arm].loc[ROWS, "AP"]
    comparison[f"{arm} mean of 3 seeds"] = mean
    comparison[f"{arm} seed range"] = [f"{a:.3f}-{b:.3f}" for a, b in zip(low, high)]
comparison[f"{CLASSICAL_512} (deployment)"] = results[CLASSICAL_512].loc[ROWS, "AP"]
display(comparison)

gains = pd.DataFrame({name: results[name].loc[cm.OUTPUTS, "gain"] for name in (CLASSICAL, CNN, HYBRID, CLASSICAL_512)})
by_type = gains.loc[MODELLED_FLAWS].groupby(pd.Series(FLAW_TYPE)).mean()
f1 = pd.DataFrame({name: results[name].loc[ROWS, "F1"] for name in (CLASSICAL, CNN, HYBRID, CLASSICAL_512)})
print("Gain by output"); display(gains)
print("Mean gain by flaw type"); display(by_type)
print("F1 @ 0.5"); display(f1)

sections.append(("Comparison table (val AP)", (
    "Same val partition (3,875 images, >=2-vote labels) and the same metrics code in every cell. CNN "
    "and hybrid: seed-42 entry, mean and range over seeds 42-44.\n\n" + rp.df_to_markdown(comparison.round(3))
    + "\n\nGain = (AP - base rate) / (1 - base rate):\n\n" + rp.df_to_markdown(gains.round(3))
    + "\n\nMean gain by flaw type:\n\n" + rp.df_to_markdown(by_type.round(3))
    + "\n\nF1 at threshold 0.5:\n\n" + rp.df_to_markdown(f1.round(3)))))

,type,base rate,Classical GB @ 224,CNN fine-tuned (seed 42),CNN fine-tuned mean of 3 seeds,CNN fine-tuned seed range,Hybrid (seed 42),Hybrid mean of 3 seeds,Hybrid seed range,Classical GB @ 512 (deployment)
BLR,photometric,0.422,0.809,0.885,0.882,0.878-0.885,0.885,0.884,0.882-0.886,0.774
BRT,photometric,0.058,0.472,0.564,0.573,0.564-0.585,0.588,0.593,0.583-0.608,0.451
DRK,photometric,0.063,0.551,0.623,0.629,0.623-0.634,0.630,0.622,0.618-0.630,0.541
FRM,semantic,0.554,0.698,0.823,0.823,0.815-0.831,0.830,0.826,0.818-0.831,0.698
OBS,semantic,0.035,0.126,0.563,0.534,0.518-0.563,0.550,0.538,0.531-0.550,0.107
ROT,semantic,0.171,0.238,0.696,0.695,0.689-0.701,0.691,0.695,0.689-0.704,0.230
UNREC,recognisability,0.156,0.730,0.802,0.798,0.788-0.805,0.797,0.795,0.788-0.801,0.729
macro (6 flaws),,0.217,0.482,0.692,0.689,0.687-0.692,0.695,0.693,0.691-0.695,0.467


Gain by output


,Classical GB @ 224,CNN fine-tuned,Hybrid,Classical GB @ 512
BLR,0.669,0.800,0.800,0.609
BRT,0.440,0.537,0.563,0.417
DRK,0.521,0.597,0.605,0.510
FRM,0.322,0.603,0.618,0.322
OBS,0.094,0.547,0.533,0.074
ROT,0.080,0.633,0.627,0.070
UNREC,0.680,0.765,0.759,0.679


Mean gain by flaw type


,Classical GB @ 224,CNN fine-tuned,Hybrid,Classical GB @ 512
photometric,0.543,0.645,0.656,0.512
semantic,0.165,0.594,0.593,0.155


F1 @ 0.5


,Classical GB @ 224,CNN fine-tuned,Hybrid,Classical GB @ 512
BLR,0.728,0.792,0.791,0.715
BRT,0.359,0.410,0.407,0.334
DRK,0.362,0.406,0.405,0.371
FRM,0.658,0.759,0.762,0.666
OBS,0.146,0.290,0.302,0.132
ROT,0.329,0.632,0.607,0.324
UNREC,0.610,0.642,0.653,0.611
macro (6 flaws),0.430,0.548,0.546,0.424


## 3. The pre-registered 224 vs 512 rule
The rule (docs/decisions.md, 2026-10-04) anticipated 224 = 512 on BLR (resolution not the driver)
or 512 better (blur needs detail 224 px removes). Stage C found 224 clearly better, outside the
rule; it was recorded as a finding and the rule was not amended. Restated here on the final table.

In [5]:
rule = pd.DataFrame({CLASSICAL: results[CLASSICAL].loc[["BLR", mt.MACRO], "AP"],
                     CLASSICAL_512: results[CLASSICAL_512].loc[["BLR", mt.MACRO], "AP"]})
rule["224 - 512"] = rule[CLASSICAL] - rule[CLASSICAL_512]
display(rule)
rule_diff = mt.paired_difference(targets, scores[CLASSICAL], scores[CLASSICAL_512], N_RESAMPLES)
display(rule_diff.loc[["BLR", mt.MACRO]])
sections.append(("Pre-registered 224 vs 512 rule", (
    "Classical GB, BLR and macro AP, with the paired bootstrap interval of the 224 - 512 difference:"
    "\n\n" + rp.df_to_markdown(rule.round(3)) + "\n\n"
    + rp.df_to_markdown(rule_diff.loc[["BLR", mt.MACRO]].round(3)))))

,Classical GB @ 224,Classical GB @ 512,224 - 512
BLR,0.809,0.774,0.035
macro (6 flaws),0.482,0.467,0.015


,difference,low,high,excludes 0
BLR,0.035,0.024,0.045,True
macro (6 flaws),0.015,0.009,0.022,True


## 4. Bootstrap intervals
95% percentile intervals from 1,000 resamples of the val images (seed 42), with every model scored
on the same resampled images, so differences are paired. They show how much each AP depends on
which images happen to be in the val partition; training randomness is the separate seed spread.
An interval of a difference that excludes 0 means the gap is larger than that sampling variation.

In [6]:
intervals = mt.ap_with_interval(targets, {n: scores[n] for n in (CLASSICAL, CNN, HYBRID)}, N_RESAMPLES)
ci = pd.DataFrame({n: [f"{r.AP:.3f} [{r.low:.3f}, {r.high:.3f}]" for r in t.itertuples()]
                   for n, t in intervals.items()}, index=intervals[CNN].index)
print("AP [95% interval]"); display(ci)

pairs = {"CNN - classical": (CNN, CLASSICAL), "hybrid - CNN": (HYBRID, CNN)}
differences = {label: mt.paired_difference(targets, scores[a], scores[b], N_RESAMPLES) for label, (a, b) in pairs.items()}
for label, table in differences.items():
    print(label); display(table)
sections.append(("Bootstrap intervals (val)", (
    "AP with 95% bootstrap interval (1,000 resamples of the val images, seed 42):\n\n" + rp.df_to_markdown(ci)
    + "".join(f"\n\nPaired difference, {label}:\n\n" + rp.df_to_markdown(t.round(3))
              for label, t in differences.items()))))

AP [95% interval]


,Classical GB @ 224,CNN fine-tuned,Hybrid
BLR,"0.809 [0.790, 0.828]","0.885 [0.872, 0.898]","0.885 [0.871, 0.899]"
BRT,"0.472 [0.404, 0.542]","0.564 [0.496, 0.626]","0.588 [0.517, 0.651]"
DRK,"0.551 [0.484, 0.607]","0.623 [0.562, 0.677]","0.630 [0.564, 0.685]"
FRM,"0.698 [0.676, 0.723]","0.823 [0.806, 0.840]","0.830 [0.813, 0.847]"
OBS,"0.126 [0.092, 0.179]","0.563 [0.491, 0.639]","0.550 [0.471, 0.632]"
ROT,"0.238 [0.217, 0.267]","0.696 [0.660, 0.732]","0.691 [0.652, 0.730]"
UNREC,"0.730 [0.700, 0.763]","0.802 [0.776, 0.828]","0.797 [0.770, 0.823]"
macro (6 flaws),"0.482 [0.464, 0.502]","0.692 [0.672, 0.714]","0.695 [0.674, 0.717]"


CNN - classical


,difference,low,high,excludes 0
BLR,0.076,0.061,0.091,True
BRT,0.092,0.033,0.145,True
DRK,0.072,0.036,0.105,True
FRM,0.125,0.105,0.144,True
OBS,0.437,0.353,0.506,True
ROT,0.458,0.422,0.493,True
UNREC,0.072,0.053,0.092,True
macro (6 flaws),0.210,0.189,0.227,True


hybrid - CNN


,difference,low,high,excludes 0
BLR,-5.224e-05,-4.240e-03,0.004,False
BRT,2.391e-02,1.984e-03,0.046,True
DRK,7.167e-03,-1.372e-02,0.026,False
FRM,6.499e-03,-8.794e-04,0.013,False
OBS,-1.311e-02,-3.957e-02,0.009,False
ROT,-5.233e-03,-1.785e-02,0.008,False
UNREC,-5.152e-03,-1.191e-02,0.002,False
macro (6 flaws),3.197e-03,-3.963e-03,0.010,False


## 5. Ablation 1: training labels at >=1 vote
At >=1 vote the rare flaws gain many positives (OBS 840 -> 3,268 on train) at the cost of noisier
labels. Training flaw labels are switched to >=1 vote; UNREC stays at >=2, and scoring (and, for
the CNN, early stopping) stays at >=2, so the task measured is unchanged. The classical GB is refit
here (seconds); the CNN run comes from `scripts/train_cnn_ablations.py`.

In [7]:
train_records = splits["train"]
train_counts = pd.DataFrame({">= 2 votes": cm.build_targets(train_records).sum(),
                             ">= 1 vote": cm.build_targets(train_records, flaw_threshold=1).sum()})
train_counts["ratio"] = train_counts[">= 1 vote"] / train_counts[">= 2 votes"]
print("Training positives"); display(train_counts)

features = cm.load_features(DATA_DIR, 224)
train_targets_votes1 = cm.build_targets(train_records, flaw_threshold=1)
models_votes1 = cm.train_models(features.loc[train_targets_votes1.index], train_targets_votes1, "gb")
scores[CLASSICAL_VOTES1] = cm.predict_scores(models_votes1, features.loc[targets.index])
scores[CLASSICAL_VOTES1].to_csv(PREDICTIONS_DIR / "classical_gb_224_votes1_val.csv")
results[CLASSICAL_VOTES1] = evaluate_with_gain(scores[CLASSICAL_VOTES1])

votes = pd.DataFrame({"base rate": base_rate,
                      f"{CLASSICAL} (>= 2)": results[CLASSICAL].loc[ROWS, "AP"],
                      CLASSICAL_VOTES1: results[CLASSICAL_VOTES1].loc[ROWS, "AP"],
                      f"{CNN} (>= 2)": results[CNN].loc[ROWS, "AP"],
                      CNN_VOTES1: results[CNN_VOTES1].loc[ROWS, "AP"]})
print("Val AP, scored at >= 2 votes"); display(votes)
votes_diff = {"classical: >= 1 - >= 2": mt.paired_difference(targets, scores[CLASSICAL_VOTES1], scores[CLASSICAL], N_RESAMPLES),
              "CNN: >= 1 - >= 2": mt.paired_difference(targets, scores[CNN_VOTES1], scores[CNN], N_RESAMPLES)}
for label, table in votes_diff.items():
    print(label); display(table)
sections.append(("Ablation 1: training labels at >=1 vote (val, scored at >=2)", (
    "Training positives:\n\n" + rp.df_to_markdown(train_counts.round(2)) + "\n\nVal AP:\n\n"
    + rp.df_to_markdown(votes.round(3))
    + "".join(f"\n\nPaired difference, {label}:\n\n" + rp.df_to_markdown(t.round(3))
              for label, t in votes_diff.items()))))

Training positives


,>= 2 votes,>= 1 vote,ratio
BLR,9552,14183,1.485
BRT,1248,4151,3.326
DRK,1367,4331,3.168
FRM,12934,17713,1.369
OBS,840,3268,3.890
ROT,3986,6699,1.681
UNREC,3558,3558,1.000


  [gb] BLR    14183 positives  fitted in   0.6s
  [gb] BRT     4151 positives  fitted in   0.4s
  [gb] DRK     4331 positives  fitted in   0.3s
  [gb] FRM    17713 positives  fitted in   0.5s
  [gb] OBS     3268 positives  fitted in   0.5s
  [gb] ROT     6699 positives  fitted in   0.6s
  [gb] UNREC   3558 positives  fitted in   0.7s
Val AP, scored at >= 2 votes


,base rate,Classical GB @ 224 (>= 2),"Classical GB @ 224, flaws >= 1 vote",CNN fine-tuned (>= 2),"CNN, flaws >= 1 vote"
BLR,0.422,0.809,0.808,0.885,0.878
BRT,0.058,0.472,0.468,0.564,0.596
DRK,0.063,0.551,0.556,0.623,0.600
FRM,0.554,0.698,0.696,0.823,0.800
OBS,0.035,0.126,0.110,0.563,0.523
ROT,0.171,0.238,0.238,0.696,0.680
UNREC,0.156,0.730,0.730,0.802,0.801
macro (6 flaws),0.217,0.482,0.479,0.692,0.679


classical: >= 1 - >= 2


,difference,low,high,excludes 0
BLR,-7.084e-04,-0.008,0.007,False
BRT,-4.539e-03,-0.025,0.016,False
DRK,5.462e-03,-0.010,0.023,False
FRM,-1.867e-03,-0.011,0.008,False
OBS,-1.570e-02,-0.043,0.008,False
ROT,5.948e-04,-0.020,0.020,False
UNREC,0.000e+00,0.000,0.000,False
macro (6 flaws),-2.793e-03,-0.010,0.005,False


CNN: >= 1 - >= 2


,difference,low,high,excludes 0
BLR,-0.007,-0.013,-0.001,True
BRT,0.032,-0.008,0.073,False
DRK,-0.023,-0.050,0.004,False
FRM,-0.023,-0.032,-0.013,True
OBS,-0.040,-0.082,0.002,False
ROT,-0.016,-0.033,0.001,False
UNREC,-0.001,-0.011,0.008,False
macro (6 flaws),-0.013,-0.024,-0.002,True


## 6. Ablation 2: ImageNet weights vs random initialisation
The same network and recipe, starting from random weights. The recipe (learning rates, at most 15
epochs) was set for a pretrained network; a from-scratch network would do somewhat better with a
longer, tuned schedule, so the gap is an upper bound on what pretraining contributes under this
recipe.

In [8]:
def ablation_table(a, b):
    table = pd.DataFrame({"type": [FLAW_TYPE.get(o, "") for o in ROWS], "base rate": base_rate,
                          f"AP {a}": results[a].loc[ROWS, "AP"], f"AP {b}": results[b].loc[ROWS, "AP"],
                          f"gain {a}": results[a].loc[ROWS, "gain"], f"gain {b}": results[b].loc[ROWS, "gain"]})
    by_type = table.loc[MODELLED_FLAWS].groupby("type")[[f"gain {a}", f"gain {b}"]].mean()
    return table, by_type


def run_summary(names):
    rows = {}
    for name in names:
        h = json.loads((RUNS_DIR / RUN_DIRS[name] / "history.json").read_text())
        e = pd.DataFrame(h["epochs"])
        rows[name] = {"epochs run": len(e), "best epoch": h["best_epoch"],
                      "best val macro AP": round(h["best_val_macro_ap"], 4),
                      "train loss at best epoch": round(e.set_index("epoch").loc[h["best_epoch"], "train_loss"], 3),
                      "torch": h["environment"]["libraries"].split("torch ")[1].split(",")[0]}
    return pd.DataFrame(rows).T


scratch, scratch_by_type = ablation_table(CNN, CNN_SCRATCH)
scratch_runs = run_summary((CNN, CNN_SCRATCH))
scratch_diff = mt.paired_difference(targets, scores[CNN], scores[CNN_SCRATCH], N_RESAMPLES)
display(scratch_runs); display(scratch); display(scratch_by_type)
print("pretrained - random init"); display(scratch_diff)
sections.append(("Ablation 2: ImageNet weights vs random initialisation (val)", (
    rp.df_to_markdown(scratch_runs) + "\n\n" + rp.df_to_markdown(scratch.round(3))
    + "\n\nMean gain by flaw type:\n\n" + rp.df_to_markdown(scratch_by_type.round(3))
    + "\n\nPaired difference, pretrained - random init:\n\n" + rp.df_to_markdown(scratch_diff.round(3)))))

,epochs run,best epoch,best val macro AP,train loss at best epoch,torch
CNN fine-tuned,6,3,0.692,0.488,2.11.0+cu130
"CNN, random init",15,13,0.517,0.665,2.14.1+cu130


,type,base rate,AP CNN fine-tuned,"AP CNN, random init",gain CNN fine-tuned,"gain CNN, random init"
BLR,photometric,0.422,0.885,0.729,0.800,0.531
BRT,photometric,0.058,0.564,0.469,0.537,0.436
DRK,photometric,0.063,0.623,0.517,0.597,0.485
FRM,semantic,0.554,0.823,0.687,0.603,0.297
OBS,semantic,0.035,0.563,0.251,0.547,0.223
ROT,semantic,0.171,0.696,0.450,0.633,0.336
UNREC,recognisability,0.156,0.802,0.725,0.765,0.674
macro (6 flaws),,0.217,0.692,0.517,NaN,NaN


,gain CNN fine-tuned,"gain CNN, random init"
type,,
photometric,0.645,0.484
semantic,0.594,0.285


pretrained - random init


,difference,low,high,excludes 0
BLR,0.155,0.137,0.173,True
BRT,0.095,0.041,0.147,True
DRK,0.105,0.069,0.142,True
FRM,0.137,0.116,0.156,True
OBS,0.312,0.226,0.389,True
ROT,0.246,0.207,0.282,True
UNREC,0.077,0.056,0.099,True
macro (6 flaws),0.175,0.154,0.193,True


## 7. Ablation 3: frozen backbone vs full fine-tuning
The Stage D runs: the frozen backbone (heads only, 5 epochs) against full fine-tuning. No new
training.

In [9]:
frozen, frozen_by_type = ablation_table(CNN, FROZEN)
frozen_runs = run_summary((CNN, FROZEN))
frozen_diff = mt.paired_difference(targets, scores[CNN], scores[FROZEN], N_RESAMPLES)
display(frozen_runs); display(frozen); display(frozen_by_type)
print("fine-tuned - frozen"); display(frozen_diff)
sections.append(("Ablation 3: frozen backbone vs full fine-tuning (val)", (
    rp.df_to_markdown(frozen_runs) + "\n\n" + rp.df_to_markdown(frozen.round(3))
    + "\n\nMean gain by flaw type:\n\n" + rp.df_to_markdown(frozen_by_type.round(3))
    + "\n\nPaired difference, fine-tuned - frozen:\n\n" + rp.df_to_markdown(frozen_diff.round(3)))))

,epochs run,best epoch,best val macro AP,train loss at best epoch,torch
CNN fine-tuned,6,3,0.692,0.488,2.11.0+cu130
CNN frozen,5,5,0.592,0.68,2.11.0+cu130


,type,base rate,AP CNN fine-tuned,AP CNN frozen,gain CNN fine-tuned,gain CNN frozen
BLR,photometric,0.422,0.885,0.809,0.800,0.669
BRT,photometric,0.058,0.564,0.412,0.537,0.376
DRK,photometric,0.063,0.623,0.545,0.597,0.514
FRM,semantic,0.554,0.823,0.795,0.603,0.540
OBS,semantic,0.035,0.563,0.354,0.547,0.330
ROT,semantic,0.171,0.696,0.637,0.633,0.562
UNREC,recognisability,0.156,0.802,0.775,0.765,0.733
macro (6 flaws),,0.217,0.692,0.592,NaN,NaN


,gain CNN fine-tuned,gain CNN frozen
type,,
photometric,0.645,0.520
semantic,0.594,0.478


fine-tuned - frozen


,difference,low,high,excludes 0
BLR,0.076,0.064,0.088,True
BRT,0.152,0.096,0.201,True
DRK,0.078,0.038,0.118,True
FRM,0.028,0.013,0.043,True
OBS,0.209,0.140,0.276,True
ROT,0.059,0.033,0.083,True
UNREC,0.027,0.011,0.043,True
macro (6 flaws),0.100,0.083,0.118,True


## 8. Ablation 4: backbone comparison (not run)
EfficientNet-B0 vs ResNet-50 is the lowest-priority ablation and the first to drop under the
compute contingency; it was not run.

In [10]:
sections.append(("Ablation 4: backbone comparison", (
    "Not run. EfficientNet-B0 vs ResNet-50 is the lowest-priority ablation in the plan and the first to "
    "drop under the compute and time contingency.")))

## 9. Write the report
The findings block in `reports/results_comparison.md` is hand-written after reading the results
and is preserved on every rerun.

In [11]:
long = pd.concat({(n if isinstance(n, str) else f"{n[0]}, seed {n[1]}"): t for n, t in results.items()
                  if not (isinstance(n, str) and n in (CNN, HYBRID))}, names=["model", "output"])
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
long.reset_index().to_csv(RESULTS_CSV, index=False, float_format="%.4f")
out = rp.write_report(REPORT_PATH, sections, title="Stage F: Comparison and ablations",
                      environment=ENVIRONMENT, findings_heading="Findings")
print("Report written to", out)
print("Results table written to", RESULTS_CSV)

Report written to /content/drive/MyDrive/Masters/Sem 3/capstone/repo/reports/results_comparison.md
Results table written to /content/drive/MyDrive/Masters/Sem 3/capstone/repo/reports/results_comparison.csv
